<h1 style="font-family: 'Georgia', serif; color: orange; font-size: 40px; text-align: center; font-weight: 500;">Importing the Required Libraries</h1>


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")



from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
import dill


from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,roc_auc_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import os

-----

<h1 style="font-family: 'Georgia', serif; color: orange; font-size: 40px; text-align: center; font-weight: 500;">Reading the Dataset</h1>


In [ ]:
data = pd.read_csv('../dataset/Churn_Modelling.csv')
data.head()

<h6 style="font-family: 'Arial'; font-size: 20px; font-style: italic; color:magenta;">Checking Columns/Features Present</h6>


In [ ]:
print(f"Number of Rows/Records in the Dataset is : {data.shape[0]}\n")
print(f"Number of Columns/Features in the Dataset is : {data.shape[1]}\n")

print(f"The Features are :\n{list(data.columns)}")

<h6 style="font-family: 'Arial'; font-size: 20px; font-style: italic; color:magenta">Inspect the first and last few rows of the dataset to understand its structure.</h6>

In [ ]:
data.head()

In [ ]:
data.tail()

<h6 style="font-family: 'Arial'; font-size: 20px; font-style: italic; color:magenta;">Random Sample of the dataset.</h6>

In [ ]:
data.sample(6)

------

<h1 style="font-family: 'Georgia', serif; color: orange; font-size: 40px; text-align: center; font-weight: 500;">Basic Inspection and Understanding Dataset</h1>


<h6 style="font-family: 'Arial'; font-size: 20px; font-style: italic; color:magenta">Examine Datatypes and Information about features</h6>

In [ ]:
# Information about the Dataset
data.info()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Observation:
Features Having numerical and categorical data type and no Datetime feature is there.</h7>

<h6 style="font-family: 'Arial'; font-size: 20px; font-style: italic; color:magenta">Check for null values if any</h6>

In [ ]:
def summarize_null_values(data):
    null_value_count = data.isnull().sum()
    null_value_percentage = np.round((null_value_count / len(data)) * 100, 2)
    null_summary = pd.DataFrame({
        'Null Value Count': null_value_count,
        'Null Value Percentage (%)': null_value_percentage
    })
    return null_summary

In [ ]:
null_summary = summarize_null_values(data)
null_summary

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Observation: There is No Missing Value Present inside the Dataset </h7>

<h6 style="font-family: 'Arial'; font-size: 20px; font-style: italic; color:magenta">Check for Duplicates if any</h6>

In [ ]:
# Checking for the Duplicate Values
data.duplicated().sum()


<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Observation: There is No Duplicate Value Present inside the dataset.</h7>

<h6 style="font-family: 'Arial'; font-size: 20px; font-style: italic; color:magenta">Examine Number of Unique Value For Each Feature/Column</h6>

In [ ]:
# Checking for the Number of Unqiue Values.
data.nunique()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Observation: Since RowNumber and CustomerId are unique identifiers for each record, they do not contribute any predictive power to the model and are irrelevant for analysis. Similarly, Surname is a personal identifier and not useful for prediction. Therefore, it's better to drop these columns.</h7>

<h6 style="font-family: 'Arial'; font-size: 20px; font-style: italic; color:magenta">Eliminating Irrelevant Features</h6> 

In [ ]:
# Columns to Drop (Irrelevant for Analysis/Prediction)
data.drop(columns=['RowNumber','CustomerId','Surname'],inplace=True)

In [ ]:
data.head()

In [ ]:
data.info()

<h6 style="font-family: 'Arial'; font-size: 20px; font-style: italic; color:magenta">DataType Conversion</h6>

### `Exited`:
- Indicates whether the customer churned (1) or retained (0).
- It’s a **binary categorical feature**.
- Convert to `object` to treat it as a category rather than a numeric value.

---

### `IsActiveMember`:
- Indicates whether the customer is active (1) or not (0).
- It’s a **binary categorical feature**.
- Convert to `object` to treat it as a category rather than a numeric value.

---

### `HasCrCard`:
- Indicates whether the customer has a credit card (1) or not (0).
- It’s a **binary categorical feature**.
- Convert to `object` to treat it as a category rather than a numeric value.

---


### `NumOfProducts`:
- Indicates the number of products the customer has (e.g., 1, 2, 3,4).
- It’s a **discrete categorical feature** representing distinct counts of products.
- Convert to `object` to treat it as a category rather than a numerical value.



In [ ]:

data['HasCrCard'] = data['HasCrCard'].astype('object')

data['IsActiveMember'] = data['IsActiveMember'].astype('object')


data['NumOfProducts'] = data['NumOfProducts'].astype('object')


data['Exited'] = data['Exited'].astype('object')

 <h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Observation: 
 Among the columns, Geography and Gender are categorical features, while the remaining columns are numerical features that represent customer attributes and financial data.</h7>



<h6 style="font-family: 'Arial'; font-size: 20px; font-style: italic; color:magenta">Segregation of Data into Numerical Columns and Categorical Columns</h6>

In [ ]:
# Segregating the Columns as Numerical and Categorical Columns

numerical_data = data.select_dtypes(include=['number'])
categorical_data = data.select_dtypes(include=['object'])

numerical_columns = list(numerical_data.columns)
categorical_columns = list(categorical_data.columns)

In [ ]:
print(f"The Numerical Columns present inside the dataset are :\n{numerical_columns}")
print("\n")
print(f"The Categorical Columns present inside the dataset are :\n{categorical_columns}")

<h6 style="font-family: 'Arial'; font-size: 20px; font-style: italic; color:magenta"> Displaying the number of unique values for each numerical column</h6>

In [ ]:
# Displaying the number of unique values for each numerical column

for col in numerical_columns:
    print(f"Column: {col} | Unique Count: {data[col].nunique()}")


<h6 style="font-family: 'Arial'; font-size: 20px; font-style: italic; color:magenta">Displaying the number of unique values and their actual categories for each categorical colum</h6>

In [ ]:
# Displaying the number of unique values and their actual categories for each categorical column

for col in categorical_columns:
    print(f"Column: {col} | Unique Count: {data[col].nunique()} | Unique Values: {list(data[col].unique())}")

    

<h6 style="font-family: 'Arial'; font-size: 20px; font-style: italic; color:magenta">Basic Statistical Summary : Summary of Numerical Features and Summary of Categorical Features</h6>

In [ ]:
# Generating basic statistical summaries for both numerical and categorical columns

numerical_data.describe().T


In [ ]:
categorical_data.describe().T

------------------

<h1 style="font-family: 'Georgia', serif; color: orange; font-size: 40px; text-align: center; font-weight: 500;">Statistical Analysis and Understanding Distributions</h1>




<h1 style="font-family: 'Georgia', serif; color:aqua; font-size:22px; font-weight: bold; text-align: center; text-transform: uppercase; letter-spacing: 2px;">Mean, Median, Mode,Range, Standard Deviation, Variance, Skewness, and Ku
</h1>


In [ ]:
# Iterating over all numerical columns 

for column in numerical_columns:
    # Calculate mean, median, and mode
    mean_value = data[column].mean()
    median_value = data[column].median()
   
    
    # Check if the mode is not empty
    if not data[column].mode().empty:
        # Extracting the first mode value
        mode_value = data[column].mode().iloc[0]
    else:
        # Set mode_value to None if no mode exists
        mode_value = None
    
    print(f"The column '{column}' has the following central tendencies:")
    print(f"Mean: {mean_value}")
    print(f"Median: {median_value}")
    print(f"Mode: {mode_value}")
    print('-' * 130)


In [ ]:
# Iterating  over all numerical columns in the DataFrame for Measure of Dispersion

for column in numerical_columns:
    
   # Calculating Range (unit will be the same as the original feature)
    range_value = data[column].max() - data[column].min()

    # Calculating Variance (unit will be the square of the original feature's unit)
    variance_value = data[column].var()

    # Calculating Standard Deviation (SD) (unit will be the same as the original feature)
    sd_value = data[column].std()

    
    print(f"The column '{column}' has the following Measures of Dispersion:")
    print(f"Range: {range_value}")
    print(f"Variance: {variance_value}")
    print(f"Standard Deviation (SD): {sd_value}")
    print('-' *100)
    
    print(f"Observations for '{column}':")
    
    # Interpreting Range
    print(f"1. Range: A high range ({range_value}) indicates that the values in '{column}' span widely.")
    
  # Interpreting Standard Deviation and Variance
    print(f"""2. Standard Deviation and Variance: Higher values of SD ({sd_value}) and Variance ({variance_value}) suggest greater variability in '{column}'.
    This means the data points are more spread out from the mean, indicating more diversity or inconsistency in the values.
    For example, a higher SD/Variance in '{column}' indicates that values can vary widely from the average, whereas lower values would suggest that most data points are concentrated around the mean.""")

    print('=' * 140)
    print('\n')


In [ ]:
# Import necessary library for kurtosis
from scipy.stats import kurtosis

In [ ]:
# Iterating over all numerical columns in the DataFrame for Skewness and Kurtosis

for column in numerical_columns:
    
    # Calculating Skewness
    skewness_value = data[column].skew()
    
    # Calculating Kurtosis
    kurtosis_value = kurtosis(data[column], fisher=True)  # Fisher=True for excess kurtosis . This means that the kurtosis value is adjusted so that the normal distribution has a kurtosis of 0.
    
    # Print the results for Skewness and Kurtosis
    print(f"The column '{column}' has the following Skewness and Kurtosis:")
    print(f"Skewness: {skewness_value}")
    print(f"Kurtosis: {kurtosis_value}")
    print('-' * 100)
    
    print(f"Observations for '{column}':")
    
    # Interpreting Skewness
    if skewness_value > 0:
        print(f"1. Skewness: Positive skewness ({skewness_value}) means '{column}' has a longer right tail (values are skewed right).")
    elif skewness_value < 0:
        print(f"1. Skewness: Negative skewness ({skewness_value}) means '{column}' has a longer left tail (values are skewed left).")
    else:
        print(f"1. Skewness: A skewness value near zero indicates symmetry in the distribution of '{column}'.")
    
    # Interpreting Kurtosis
    if kurtosis_value > 0:
        print(f"2. Kurtosis: Positive kurtosis ({kurtosis_value}) indicates a more peaked distribution (leptokurtic).")
    elif kurtosis_value < 0:
        print(f"2. Kurtosis: Negative kurtosis ({kurtosis_value}) indicates a flatter distribution (platykurtic).")
    else:
        print(f"2. Kurtosis: A kurtosis value near zero indicates a normal (mesokurtic) distribution.")
    
    print('=' * 130)
    print('\n')


<h1 style="font-family: 'Georgia', serif; color:aqua; font-size:22px; font-weight: bold; text-align: center; text-transform: uppercase; letter-spacing: 2px;">Analyze Distribution ( Numerical Data Distribution and Categorical Data Distribution)
</h1>


<span style="font-family: 'Brush Script MT', cursive; color:pink; font-size: 50px; ">Numerical Features Distribution</span>
<hr style="border: none; height: 2px; background-color: pink; width: 90%; margin: 2px auto;">


In [ ]:
print(f"We have {len(numerical_columns)} Numerical Features")

In [ ]:
import scipy.stats as stats

In [ ]:
# Function to Check Distribution

def check_distribution(column):
    """Check if the data is normally distributed using Shapiro-Wilk test."""
    column = column.dropna()  # Drop NaN values
    stat, p_value = stats.shapiro(column)
    
    print(f"Shapiro-Wilk Test:")
    print(f"Statistic: {stat:.4f}, P-value: {p_value:.4f}")
    
    if p_value > 0.05:
        print("Conclusion: The data is normally distributed (p-value > 0.05).\n")
        return True
    else:
        print("Conclusion: The data is not normally distributed (p-value ≤ 0.05).\n")
        return False


In [ ]:
# Function to Plot Distribution

def plot_distribution(data, column_name, color):
    """Plot histogram, Q-Q plot, boxplot, and KDE plot for the specified column."""
    column = data[column_name].dropna()  # Drop NaN values
    
    fig, axs = plt.subplots(2, 2, figsize=(16,7))
    fig.suptitle(f'Data Distribution of {column_name}', fontsize=16, style='italic', weight='bold', color='darkblue')

    # Histogram & KDE Plot
    sns.histplot(column, kde=True, color=color, ax=axs[0, 0], bins=30)
    axs[0, 0].set_title(f"Histogram & KDE - {column_name}", fontsize=12, style='italic', weight='bold')
    axs[0, 0].set_xlabel("Value", fontsize=10)
    axs[0, 0].set_ylabel("Frequency", fontsize=10)
    axs[0, 0].grid(True, linestyle='--', alpha=0.6)

    # Q-Q Plot
    stats.probplot(column, dist="norm", plot=axs[0, 1])
    axs[0, 1].set_title(f"Q-Q Plot - {column_name}", fontsize=12, style='italic', weight='bold')
    axs[0, 1].grid(True, linestyle='--', alpha=0.6)

    # Horizontal Boxplot
    sns.boxplot(x=column, color=color, ax=axs[1, 0])
    axs[1, 0].set_title(f"Boxplot - {column_name}", fontsize=12, style='italic', weight='bold')
    axs[1, 0].set_xlabel("Value", fontsize=10)
    axs[1, 0].grid(True, linestyle='--', alpha=0.6)

    # KDE Plot (without Histogram)
    sns.kdeplot(column, color=color, ax=axs[1, 1], fill=True)
    axs[1, 1].set_title(f"KDE Plot - {column_name}", fontsize=12, style='italic', weight='bold')
    axs[1, 1].set_xlabel("Value", fontsize=10)
    axs[1, 1].grid(True, linestyle='--', alpha=0.6)

    # Adjust layout
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()


In [ ]:
# Function to print Standard Deviation and Variance

def print_standard_deviation_variance(data, column_name):
    """Print standard deviation and variance of the data and provide conclusions."""
    column = data[column_name].dropna()  # Drop NaN values
    
    # Calculate standard deviation and variance
    std_dev = column.std()
    variance = column.var()
    
    # Print standard deviation and variance
    print(f"Standard Deviation of {column_name}: {std_dev:.2f}")
    print(f"Variance of {column_name}: {variance:.2f}")
    
    # Standard deviation conclusion
    if std_dev < 1:
        print("The data has low variability (Standard Deviation < 1). Most values are close to the mean.")
    elif 1 <= std_dev < 3:
        print("The data has moderate variability (1 <= Standard Deviation < 3). Values are spread out but not extreme.")
    else:
        print("The data has high variability (Standard Deviation >= 3). Values are widely spread out from the mean.")
    
    # Variance conclusion
    if variance < 1:
        print("The data has low variance (Variance < 1). The spread of data points is small.")
    elif 1 <= variance < 9:
        print("The data has moderate variance (1 <= Variance < 9). The spread of data points is moderate.")
    else:
        print("The data has high variance (Variance >= 9). The spread of data points is large.")


In [ ]:
# Function to Print Skewness and Kurtosis

def print_skewness_kurtosis(data, column_name):
    """Print skewness and kurtosis of the data and provide conclusions."""
    column = data[column_name].dropna()  # Drop NaN values
    skewness = stats.skew(column)
    kurtosis_value = stats.kurtosis(column)
    
    print(f"Skewness of {column_name}: {skewness:.2f}")
    print(f"Kurtosis of {column_name}: {kurtosis_value:.2f}")
    
    # Skewness conclusion
    if skewness > 0:
        print("The data is right-skewed (positive skewness). The right tail is longer or has more extreme values.")
    elif skewness < 0:
        print("The data is left-skewed (negative skewness). The left tail is longer or has more extreme values.")
    else:
        print("The data is symmetric (skewness = 0).")

    # Kurtosis conclusion
    if kurtosis_value < 0:
        print("The data is platykurtic (kurtosis < 0). Fewer extreme outliers and a flatter distribution than a normal distribution.")
    elif kurtosis_value > 0:
        print("The data is leptokurtic (kurtosis > 0). More extreme outliers and a sharper peak than a normal distribution.")
    else:
        print("The data is mesokurtic (kurtosis = 0). Normal distribution with moderate outliers.")
    

In [ ]:
# Main Function to Analyze Distribution

def analyze_distribution(data, column_name, color):
    """Analyze and visualize the distribution of the specified column."""
    print(f"\nAnalyzing Distribution of {column_name}")
    
    # Step 1: Check distribution
    print("Determining if the data follows a normal distribution...")
    is_normal = check_distribution(data[column_name])
    print(f"The data follows a normal distribution : {is_normal}")

     # Step 2: Print standard deviation and variance
    print("Calculating standard deviation and variance to understand data spread...")
    print_standard_deviation_variance(data, column_name)
    print('\n')
    
    
    # Step 3: Print skewness and kurtosis
    print("Calculating skewness and kurtosis to understand data shape...")
    print_skewness_kurtosis(data, column_name)
    print("\n")
    
        
    # Step 4: Plot distribution
    print("Plotting the distribution of the data...")
    plot_distribution(data, column_name, color)
    print("\n")


In [ ]:
color_map = sns.color_palette("Set1", len(numerical_columns))

# Analyze each numerical column
for idx, col in enumerate(numerical_columns):
    analyze_distribution(data, col, color_map[idx])
    print('_'*180)
    print('\n')

In [ ]:
color_map = sns.color_palette("Set2", len(numerical_columns))

# Define color for mean and median lines
mean_color = 'blue'
median_color = 'maroon'

# Create individual plots for each numerical column and print statistics
for idx, col in enumerate(numerical_columns):
    # Calculate mean and median
    col_mean = data[col].mean()
    col_median = data[col].median()
    
    # Print mean and median values
    print("\n")
    print(f"Mean of {col} is: {col_mean}")
    print(f"Median of {col} is: {col_median}")
    
    # Create a new figure and axis for each plot
    plt.figure(figsize=(14, 5))
    
    # Create a violin plot with a boxplot inside it
    sns.violinplot(x=data[col], color=color_map[idx], inner=None)
    sns.boxplot(x=data[col], width=0.1, color='gold', linewidth=0.5)
    
    # Plot mean and median lines
    plt.axvline(col_mean, color=mean_color, linestyle='--', label=f'Mean: {col_mean:.2f}')
    plt.axvline(col_median, color=median_color, linestyle='--', label=f'Median: {col_median:.2f}')
    
    # Add annotations
    plt.annotate(f'Mean: {col_mean:.2f}', xy=(col_mean, 0.5), xytext=(col_mean + 0.1, 0.5),
                 arrowprops=dict(facecolor=mean_color, shrink=0.05), color=mean_color)
    plt.annotate(f'Median: {col_median:.2f}', xy=(col_median, 0.5), xytext=(col_median + 0.1, 0.4),
                 arrowprops=dict(facecolor=median_color, shrink=0.05), color=median_color)
    
    # Add title and legend
    plt.title(f'Violin Plot for {col} with Mean and Median Points!', 
              fontweight='bold', fontstyle='italic', color='navy')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    print('_'*180)


<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">


<span style="font-family: 'Brush Script MT', cursive; color:pink; font-size: 50px; ">Outlier Detection and Handling</span>
<hr style="border: none; height: 2px; background-color: pink; width: 90%; margin: 2px auto;">

Since we observed outliers only in the 'CreditScore' and 'Age' columns, we will handle outliers exclusively in these two columns.


In [ ]:
def cap_outliers_iqr(df, column):
    """
    Detects and caps outliers in a given column using the IQR method.
    
    Parameters:
    df (pd.DataFrame): The dataframe containing the column
    column (str): The name of the column to process
    
    Returns:
    pd.DataFrame: Updated dataframe with outliers capped
    """
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    # Define bounds
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    print(f"\nProcessing '{column}':")
    print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
    print(f"Lower Bound: {lower_bound}, Upper Bound: {upper_bound}")

    # Count outliers before capping
    num_outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)].shape[0]
    print(f"Number of outliers before capping: {num_outliers}")

    # Capping
    df[column] = df[column].apply(lambda x: lower_bound if x < lower_bound else upper_bound if x > upper_bound else x)

    # Count after capping (should be 0)
    num_outliers_after = df[(df[column] < lower_bound) | (df[column] > upper_bound)].shape[0]
    print(f"Number of outliers after capping: {num_outliers_after}")

    return df


In [ ]:
# Apply capping on CreditScore and Age
data = cap_outliers_iqr(data, 'CreditScore')
data = cap_outliers_iqr(data, 'Age')


In [ ]:
color_map = sns.color_palette("Set1", len(numerical_columns))

# Analyze each numerical column
for idx, col in enumerate(numerical_columns):
    analyze_distribution(data, col, color_map[idx])
    print('_'*180)
    print('\n')

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<span style="font-family: 'Brush Script MT', cursive; color:pink; font-size: 50px; ">Categorical Features Distribution</span>
<hr style="border: none; height: 2px; background-color: pink; width: 90%; margin: 2px auto;">

In [ ]:
# Loop through each categorical column with <= 5 unique values
for column in categorical_columns:
    if data[column].nunique() <= 5:
        print(f"The Number of Unique elements present in the '{column}' column are: {data[column].nunique()}")

        # Get value counts and sort descending
        sorted_counts = data[column].value_counts().sort_values(ascending=False)

        # Create subplot: 1 row, 2 columns
        fig, axes = plt.subplots(1, 2, figsize=(18,7))

        # Bar Chart
        sns.countplot(data=data, x=column, order=sorted_counts.index, palette='muted', ax=axes[0])
        axes[0].set_title(f"Count Plot of {column}", fontsize=14)
        axes[0].set_ylabel('Count')
        axes[0].set_xlabel(column)
        
        # Bold Y-tick labels
        axes[0].set_yticklabels(axes[0].get_yticks(), fontweight='bold')

        # Bold X-tick labels
        axes[0].set_xticklabels(axes[0].get_xticklabels(), fontweight='bold')

        # Pie Chart
        axes[1].pie(sorted_counts.values, labels=[str(i) for i in sorted_counts.index],
                    autopct='%1.1f%%', colors=sns.color_palette('muted'),textprops={'fontsize': 14 ,'color': 'black', 'weight': 'bold'})
        axes[1].set_title(f"Pie Chart of {column}", fontsize=16)

        # Layout adjustment
        plt.suptitle(f"In-Depth Distribution of '{column}'",fontsize=20, color='Blue')
        plt.tight_layout(rect=[0, 0, 1, 0.95])  # Give space for suptitle
        plt.show()

        print('=' * 160)
        print('\n')


<span style="font-family: 'Brush Script MT', cursive; color:pink; font-size: 50px;">Correlation Uncovered</span>
<hr style="border: none; height: 2px; background-color: pink; width: 90%; margin: 2px auto;">

| Variable Type            | Method                    | Function / Tool                    | Description                                                             |
|---------------------------|----------------------------|-------------------------------------|-------------------------------------------------------------------------|
| Num ↔ Num                 | Pearson / Spearman         | `df.corr()`                        | Measures linear (Pearson) or rank-based (Spearman) relationships.       |
| Cat ↔ Cat                 | Cramér's V                 | `dython.nominal.associations()`    | Measures strength of association between categorical variables.        |
| Cat ↔ Num                 | Correlation Ratio (η²)     | `dython.nominal.associations()`    | Measures how much numeric variable varies by categories.               |
| Cat ↔ Cat (directional)   | Theil’s U (asymmetric)     | Manual / `dython.nominal.theils_u()` | Useful when you want to predict one category from another (not symmetric). |


In [ ]:
# Calculating the Pearson correlation matrix 
correlation_matrix = numerical_data.corr(method='pearson')  # or 'spearman'
print("Correlation Matrix for  Columns:")
correlation_matrix


In [ ]:
sns.set_theme(style="whitegrid")
corr_matrix = numerical_data.corr()

# Mask the upper triangle to focus on the lower half of the correlation matrix
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))


plt.figure(figsize=(14,7))
sns.heatmap(
    corr_matrix, 
    annot=True, 
    cmap='GnBu',
    fmt='.2f', 
    annot_kws={"size": 8, "weight": "bold", "family": "serif"},
    mask=mask  # Mask applied to display only the lower half of the correlation matrix
)

plt.title('Correlation Heatmap for Numerical Columns', 
          fontsize=14, fontweight='bold', family='Georgia')

plt.xticks(rotation=90, ha='right', fontsize=10, family='Georgia') 
plt.yticks(rotation=0, fontsize=10, family='Georgia')
plt.tight_layout()
plt.show()



In [ ]:
from dython.nominal import associations

# Get associations without plotting
assoc_result = associations(data, nominal_columns='auto', plot=False)
assoc_matrix = assoc_result['corr']

# Create mask for upper triangle
mask = np.triu(np.ones_like(assoc_matrix, dtype=bool))

# Plot styled heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(
    assoc_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    annot_kws={"size": 8, "weight": "bold", "family": "serif"},
    mask=mask
)

plt.title("Mixed-Type Associations Heatmap", fontsize=14, fontweight="bold", family="Georgia")
plt.xticks(rotation=90, ha="right", fontsize=10, family="Georgia")
plt.yticks(rotation=0, fontsize=10, family="Georgia")
plt.tight_layout()
plt.show()

In [ ]:
# Generate the correlation/association matrix
assoc_result = associations(data, nominal_columns='auto', plot=False)

# Extract only the correlation matrix
assoc_matrix = assoc_result['corr']

# Mask the upper triangle (keep only lower triangle)
mask = np.triu(np.ones_like(assoc_matrix, dtype=bool))
triangular_assoc = assoc_matrix.where(~mask)

# Flatten the triangular matrix
assoc_pairs = triangular_assoc.stack().reset_index()
assoc_pairs.columns = ['Column_1', 'Column_2', 'Association_Score']

# Drop self-correlations (diagonal elements)
assoc_pairs = assoc_pairs[assoc_pairs['Column_1'] != assoc_pairs['Column_2']]

# Sort by absolute association score
assoc_pairs_sorted = assoc_pairs.loc[
    assoc_pairs['Association_Score'].abs().sort_values(ascending=False).index
]

# Top 10 strongest associations
top_associations = assoc_pairs_sorted.head(10).reset_index(drop=True)

top_associations

-----------

<h1 style="font-family: 'Georgia', serif; color: orange; font-size: 40px; text-align: center; font-weight: 500;">Exploratory Data Analysis</h1>


In [ ]:
blue_color = 'oceanblue' 
green_color = 'limegreen'  
sns.set_palette([blue_color, green_color])

<span style="font-family: 'Arial', cursive; color:aqua ; font-size: 30px; display: block; text-align: center;">UNIVARIATE ANALYSIS</span>
<hr style="border: none; height: 3px; background-color: aqua; width: 80%; margin: 10px auto;">


<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;"> What is the age distribution of our customer base?</h2>

In [ ]:
# Age Distribution
plt.figure(figsize=(18,5))
data['Age'].plot(kind='hist', bins=30, edgecolor='black')
plt.title('Age Distribution')
plt.xlabel('Age')
plt.ylabel('Frequency')
plt.show()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Majority of customers are between 30-40 years. Tailor services for this age group.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Are our customers financially stable?</h2>

In [ ]:
# Credit Score Distribution
plt.figure(figsize=(18,5))
sns.histplot(data['CreditScore'], bins=30, kde=True)
plt.title('Credit Score Distribution')
plt.xlabel('Credit Score')
plt.show()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Most customers have credit scores between 600–700, indicating moderate financial stability.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Are we attracting customers with high income?</h2>

In [ ]:
# Estimated Salary Distribution
plt.figure(figsize=(18,5))
sns.histplot(data['EstimatedSalary'], bins=30, kde=True)
plt.title('Estimated Salary Distribution')
plt.xlabel('Salary')
plt.show()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Majority earn under 100k, signaling opportunity to target higher-income brackets.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">How many customers maintain high account balances?</h2>

In [ ]:
# Balance Distribution
plt.figure(figsize=(18,5))
sns.histplot(data['Balance'], bins=30, kde=True)
plt.title('Account Balance Distribution')
plt.xlabel('Balance')
plt.show()
print("Insight: Many accounts have zero balance; engagement or product fit may be low.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Many accounts have zero balance; engagement or product fit may be low.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Which regions have the most customers?</h2>

In [ ]:
# Geography Distribution
plt.figure(figsize=(18,5))
sns.countplot(x='Geography', data=data)
plt.title('Geographical Distribution')
plt.show()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: France leads, followed by Spain and Germany. Marketing can be localized accordingly.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<span style="font-family: 'Arial', cursive; color: aqua; font-size: 30px; display: block; text-align: center;">BIVARIATE ANALYSIS</span>
<hr style="border: none; height: 3px; background-color: aqua; width: 90%; margin: 10px auto;">


<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Are certain age groups more likely to exit?</h2>

In [ ]:
# Age vs. Exited
plt.figure(figsize=(18,5))
sns.boxplot(x='Exited', y='Age', data=data)
plt.title('Age vs. Exited')
plt.show()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Exiting customers tend to be older, suggesting loyalty programs for seniors.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Does account balance impact customer exit?</h2>

In [ ]:
# Balance vs. Exited
plt.figure(figsize=(18,5))
sns.boxplot(x='Exited', y='Balance', data=data)
plt.title('Balance vs. Exited')
plt.show()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: High balance accounts have higher exit rates, implying dissatisfaction despite value.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Do high earners exit more?</h2>

In [ ]:

# EstimatedSalary vs. Exited
plt.figure(figsize=(18,5))
sns.violinplot(x='Exited', y='EstimatedSalary', data=data)
plt.title('Estimated Salary vs. Exited')
plt.show()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Salary alone doesn't drive churn—strategies must go beyond income tiers.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Is churn rate different across regions?</h2>

In [ ]:
# Geography vs. Exited
plt.figure(figsize=(18,5))
sns.barplot(x='Geography', y='Exited', data=data)
plt.title('Geography vs. Exit Rate')
plt.show()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Germany shows higher churn. Localized improvements are needed.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Are men or women more likely to churn?</h2>

In [ ]:
# Gender vs. Exited
plt.figure(figsize=(18,5))
sns.barplot(x='Gender', y='Exited', data=data)
plt.title('Gender vs. Exit Rate')
plt.show()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Female churn is slightly higher; gender-targeted outreach can help.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">How does Age relate to Credit Score, and does it affect exit behavior?</h2>

In [ ]:
plt.figure(figsize=(18,5))
sns.scatterplot(x='Age', y='CreditScore', data=data, hue='Exited', palette='coolwarm')
plt.title('Age vs Credit Score')
plt.xlabel('Age')
plt.ylabel('Credit Score')
plt.show()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Younger customers tend to have lower credit scores, but they don't show a strong correlation with exit behavior.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">How does the number of products impact account balances?</h2>

In [ ]:
plt.figure(figsize=(18,5))
sns.boxplot(x='NumOfProducts', y='Balance', data=data)
plt.title('Balance by Number of Products')
plt.xlabel('Number of Products')
plt.ylabel('Balance')
plt.show()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Customers with more products tend to have higher balances, but they also show more variability in account balance.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Are there differences in exit rates between men and women?</h2>

In [ ]:
plt.figure(figsize=(18,5))
sns.countplot(x='Gender', hue='Exited', data=data)
plt.title('Gender vs Exited')
plt.xlabel('Gender')
plt.ylabel('Exited Count')
plt.show()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Both genders show a similar level of exit, but women tend to churn slightly more.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">How does Tenure relate to Credit Score?</h2>

In [ ]:
plt.figure(figsize=(18,5))
sns.scatterplot(x='Tenure', y='CreditScore', data=data)
plt.title('Tenure vs Credit Score')
plt.xlabel('Tenure')
plt.ylabel('Credit Score')
plt.show()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">
Insight: Credit scores appear fairly consistent across all tenure levels, suggesting tenure does not significantly influence creditworthiness in this dataset.
</h7>


<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Is there any noticeable relationship between Age and Estimated Salary?</h2>

In [ ]:
plt.figure(figsize=(18,5))
sns.scatterplot(x='Age', y='EstimatedSalary', data=data)
plt.title('Age vs Estimated Salary')
plt.xlabel('Age')
plt.ylabel('Estimated Salary')
plt.show()
print("Insight: There seems to be no strong correlation between age and estimated salary, but younger people are more likely to have lower salaries.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: There seems to be no strong correlation between age and estimated salary, but younger people are more likely to have lower salaries.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<span style="font-family: 'Arial', cursive; color: aqua; font-size: 30px; display: block; text-align: center;">MULTIVARIATE ANALYSIS</span>

<hr style="border: none; height: 3px; background-color: aqua; width: 100%; margin: 10px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">How do age and balance together influence exit?</h2>

In [ ]:
# Age vs. Balance vs. Exited
plt.figure(figsize=(18,5))
sns.scatterplot(x='Age', y='Balance', hue='Exited', data=data)
plt.title('Age vs. Balance Colored by Exit')
plt.show()
print("Insight: Older customers with higher balances tend to churn more.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Older customers with higher balances tend to churn more.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">How do credit and salary together relate to churn?</h2>

In [ ]:
# CreditScore vs. EstimatedSalary by Exited
plt.figure(figsize=(18,5))
sns.scatterplot(x='CreditScore', y='EstimatedSalary', hue='Exited', data=data)
plt.title('Credit Score vs. Estimated Salary Colored by Exit')
plt.show()
print("Insight: Churn is dispersed, not focused on low-credit or low-salary groups.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Churn is dispersed, not focused on low-credit or low-salary groups.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Does geography and gender combination affect churn?</h2>

In [ ]:
# Geography, Gender, and Exit
plt.figure(figsize=(18,5))
sns.catplot(x='Geography', hue='Gender', col='Exited', kind='count', data=data)
plt.show()
print("Insight: German female customers show higher exits. Focus efforts here.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight:France and German female customers show higher exits. Focus efforts here.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Does product engagement and credit card usage affect retention?</h2>

In [ ]:
# NumOfProducts vs. HasCrCard vs. Exited
plt.figure(figsize=(18,5))
sns.catplot(x='NumOfProducts', hue='HasCrCard', col='Exited', kind='count', data=data)
plt.show()
print("Insight: Customers with 3+ products are more likely to churn, regardless of card ownership.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Customers with 3+ products are more likely to churn, regardless of card ownership.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Does age modify churn among inactive members?</h2>

In [ ]:
# IsActiveMember vs. Age vs. Exited
plt.figure(figsize=(18,5))
sns.boxplot(x='IsActiveMember', y='Age', hue='Exited', data=data)
plt.title('IsActiveMember vs. Age vs. Exit')
plt.show()
print("Insight: Inactive older members churn more. Activation strategies needed.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Inactive older members churn more. Activation strategies needed.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">What are the relationships between Age, Credit Score, and Estimated Salary?</h2>

In [ ]:

sns.pairplot(data.assign(Exited_int=data['Exited'].astype(int))[
    ['Age', 'CreditScore', 'EstimatedSalary', 'Exited_int']],
    hue='Exited_int',
    palette='coolwarm'
)
plt.suptitle('Age, Credit Score, and Salary Distribution', fontsize=16, y=1.02)
plt.show()
print("Insight: There is a slight trend of younger individuals having lower credit scores and estimated salaries, but no strong pattern of churn.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: There is a slight trend of younger individuals having lower credit scores and estimated salaries, but no strong pattern of churn.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">What is the correlation between Balance, Number of Products, and Exited status?</h2>

In [ ]:
plt.figure(figsize=(18,5))
sns.heatmap(data[['Balance', 'NumOfProducts', 'Exited']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Between Balance, Products, and Exited')
plt.show()
print("Insight: Balance and Number of Products are positively correlated. However, an increase in the number of products doesn't always correlate to lower exit rates.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Balance and Number of Products are negatively correlated.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">How do Age and Tenure affect Exited status?</h2>

In [ ]:
plt.figure(figsize=(18,5))
sns.heatmap(data[['Age', 'Tenure', 'Exited']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Between Age, Tenure, and Exited')
plt.show()
print("Insight: Tenure and Age are negatively correlated with Exited status, indicating that older, long-tenured customers are less likely to churn.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Tenure is negatively correlated with Exited status and Age is positively correlated with Exited, indicating that long-tenured customers are less likely to churn and Older are more likely to churn.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Does gender and age affect the exit rate?</h2>

In [ ]:
plt.figure(figsize=(18,5))
sns.violinplot(x='Gender', y='Age', hue='Exited', split=True, data=data)
plt.title('Gender, Age, and Exited')
plt.xlabel('Gender')
plt.ylabel('Age')
plt.show()
print("Insight: Female customers tend to be younger and show slightly higher exit rates. The relationship between gender and exit behavior is nuanced.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Female customers tend to be younger and show slightly higher exit rates. The relationship between gender and exit behavior is nuanced.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">How do the number of products, balance, and salary impact exit probability?</h2>

In [ ]:

sns.pairplot(
    data[['NumOfProducts', 'Balance', 'EstimatedSalary', 'Exited']],
    hue='Exited',
    palette='coolwarm'
)
plt.suptitle('Relationship Between Products, Balance, and Salary with Customer Exit', fontsize=16, y=1.02)
plt.show()


print("Insight: Customers with a higher number of products have higher exit probabilities, but balance and salary alone do not drive exit behaviors.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Customers with a higher number of products have higher exit probabilities, but balance and salary alone do not drive exit behaviors.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">How does Tenure, Products, and Exit Probability interact?</h2>

In [ ]:
plt.figure(figsize=(16,8))
sns.barplot(x='Tenure', y='Exited', hue='NumOfProducts', data=data, palette='coolwarm')

plt.title('Exit Probability by Tenure and Number of Products', fontsize=14)
plt.xlabel('Customer Tenure (Bucketed)', fontsize=12)
plt.ylabel('Exit Rate', fontsize=12)
plt.legend(title='Number of Products', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print("Insight: New customers with fewer products show higher exit rates. Long-term engagement and multiple products are key to retaining customers.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: New customers with fewer products show higher exit rates. Long-term engagement and multiple products are key to retaining customers.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">How do Age, Credit Score, and Balance together influence Exited?</h2>

In [ ]:
import plotly.express as px

fig = px.scatter_3d(
    data_frame=data,
    x='Age',
    y='CreditScore',
    z='Balance',
    color='Exited',
    color_continuous_scale='RdBu',
    title='3D Plot of Age, Credit Score, and Balance',
    labels={
        'Age': 'Customer Age',
        'CreditScore': 'Credit Score',
        'Balance': 'Account Balance',
        'Exited': 'Churn Status'
    }
)

fig.update_traces(marker=dict(size=5))
fig.update_layout(scene=dict(
    xaxis_title='Age',
    yaxis_title='Credit Score',
    zaxis_title='Balance'
))
fig.show()


<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">
Insight: Churn patterns are dispersed, but slight trends suggest customers with lower balances and credit scores may be more prone to exit. Age alone doesn't clearly differentiate churn behavior.
</h7>


<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<span style="font-family: 'Arial', cursive; color: aqua; font-size: 30px; display: block; text-align: center;"> KPI-focused Visualizations</span>


<hr style="border: none; height: 3px; background-color: aqua; width: 100%; margin: 10px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Overall Churn Rate</h2>

In [ ]:
# Calculate churn and retention rates
churn_rate = data['Exited'].mean()
retention_rate = 1 - churn_rate

# Plot
labels = ['Churned', 'Retained']
sizes = [churn_rate, retention_rate]
colors = ['#FF6B6B', '#4CAF50']

fig, ax = plt.subplots(figsize=(19,6))
wedges, texts, autotexts = ax.pie(sizes, labels=labels, autopct='%1.1f%%',
                                  startangle=90, colors=colors, wedgeprops=dict(width=0.4))

# Add center circle for donut shape
centre_circle = plt.Circle((0, 0), 0.70, fc='white')
fig.gca().add_artist(centre_circle)

# Title and style
plt.title('Customer Churn Rate', fontsize=16)
plt.tight_layout()
plt.show()


print(f"Churn Rate: {churn_rate * 100:.2f}%")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">
Insight: This chart shows the proportion of customers who exited. A lower churn rate indicates higher retention success.</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Churn by Geography</h2>

In [ ]:
plt.figure(figsize=(18,5))
print("\nKPI: Churn Rate by Geography")
geo_churn = data.groupby('Geography')['Exited'].mean().sort_values(ascending=False) * 100
print(geo_churn)
geo_churn.plot(kind='bar', title='Churn Rate by Geography')
plt.ylabel('% Churn')
plt.show()


<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Germany has the highest churn. Focused campaigns needed there.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Churn by Tenure Bucket</h2>

In [ ]:
plt.figure(figsize=(18,5))
print("\nKPI: Churn by Customer Tenure")
data['TenureBucket'] = pd.cut(data['Tenure'], bins=[-1,2,5,10], labels=['New','Mid','Loyal'])
tenure_churn = data.groupby('TenureBucket')['Exited'].mean() * 100
print(tenure_churn)
tenure_churn.plot(kind='bar', title='Churn Rate by Tenure')
plt.ylabel('% Churn')
plt.show()

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: New customers churn more. Improve onboarding experience.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Revenue at Risk by Churned Customers</h2>

In [ ]:
# Total and at-risk revenue
plt.figure(figsize=(18,5))

total_balance = data['Balance'].sum()
revenue_at_risk = data[data['Exited'] == 1]['Balance'].sum()
retained_revenue = total_balance - revenue_at_risk

# Plot values
labels = ['Retained Revenue', 'Revenue at Risk']
values = [retained_revenue, revenue_at_risk]
colors = ['#4CAF50', '#FF6B6B']


plt.figure(figsize=(18, 4))
plt.barh(labels, values, color=colors)
plt.xlabel('Revenue ($)', fontsize=12)
plt.title('Revenue Distribution: Retained vs. At-Risk', fontsize=14)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print(f"Revenue at Risk: ${revenue_at_risk:,.2f}")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">
Insight: A significant portion of revenue is at stake due to churn. Focus on high-balance customers for retention.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">Product Penetration Rate (Customers with >1 product)</h2>

In [ ]:
# Calculate product penetration
penetration_data = data['NumOfProducts'].apply(lambda x: 'More than 1' if x > 1 else '1 or Less')
penetration_counts = penetration_data.value_counts()

# Plot
plt.figure(figsize=(18,5))
sns.barplot(x=penetration_counts.index, y=penetration_counts.values, palette='coolwarm')
plt.title('Product Penetration: More Than 1 vs 1 or Less', fontsize=14)
plt.xlabel('Product Count', fontsize=12)
plt.ylabel('Number of Customers', fontsize=12)
plt.tight_layout()
plt.show()


<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Low penetration suggests an opportunity for upselling additional products to increase customer value.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">What is the churn rate by Age group?</h2>

In [ ]:
plt.figure(figsize=(18,5))
sns.histplot(data=data, x='Age', hue='Exited', multiple='stack', kde=True)
plt.title('Churn by Age')
plt.xlabel('Age')
plt.ylabel('Churn Count')
plt.show()
print("Insight: Churn rates are higher among younger age groups. Targeted retention efforts for these groups are needed.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Churn rates are higher among mid to old age groups. Targeted retention efforts for these groups are needed.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">How does account balance impact exit rates?</h2>

In [ ]:
plt.figure(figsize=(18,5))
sns.boxplot(x='Exited', y='Balance', data=data)
plt.title('Balance vs Exited')
plt.xlabel('Exited')
plt.ylabel('Balance')
plt.show()
print("Insight: Higher balances tend to have lower exit rates, but some high balance customers still exit, indicating dissatisfaction.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Higher balances tend to have lower exit rates, but some high balance customers still exit, indicating dissatisfaction.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">How does Age and Credit Score affect churn probability?</h2>

In [ ]:

plt.figure(figsize=(18,5))
sns.swarmplot(x='Exited', y='Age', hue='CreditScore', data=data, palette='coolwarm', dodge=True)
plt.title('Age vs. Exit Status with Credit Score Grouping', fontsize=14)
plt.xlabel('Exited')
plt.ylabel('Age')
plt.legend(title='Credit Score')
plt.tight_layout()
plt.show()


<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">



<h2 style="font-family: 'Arial', sans-serif; color:rgb(180, 88, 238); font-size: 30px; text-align: left; font-weight: 400;">What is the exit rate based on the number of products owned?</h2>

In [ ]:
plt.figure(figsize=(18,5))
sns.barplot(x='NumOfProducts', y='Exited', data=data)
plt.title('Exit Rate by Number of Products')
plt.xlabel('Number of Products')
plt.ylabel('Exited Rate')
plt.show()
print("Insight: Customers with fewer products tend to exit more often, indicating an opportunity to upsell products to improve retention.")

<h7 style="color:#39FF14;font-style:italic; font-family:Georgia">Insight: Customers with 3+ products tend to exit more often, indicating an opportunity to upsell products to improve retention.
</h7>

<hr style="border: none; height: 2px; background-color: yellow; width: 100%; margin: 2px auto;">

--------

<h1 style="font-family: 'Georgia', serif; color: orange; font-size: 40px; text-align: center; font-weight: 500;">Feature Engineering</h1>


We use a Pipeline to streamline the preprocessing steps, including Label Encoding, Scaling, and applying a ColumnTransformer to handle numerical and categorical features separately. Since the dataset contains no missing values, we do not include any imputation step. If the dataset is imbalanced, SMOTE (Synthetic Minority Over-sampling Technique) can be applied after preprocessing and before model training to balance the target classes.

In [ ]:
data.head()

In [ ]:
# Typecasting the Target Feature:
data["Exited"] = data["Exited"].astype(int)

<h2 style="font-family: 'Brush Script MT', cursive; color: goldenrod;">Segregating Features (X) and Target (y)</h2>

In [ ]:
# Segregating the Input Features and the Target Feature:
X = data.drop(columns=['Exited'],axis=1)
y = data['Exited']

In [ ]:
# Columns Groups:
categorical_cols_for_pipeline = ['Geography', 'Gender']
numerical_cols_for_pipeline = ['CreditScore', 'Age', 'Tenure', 'Balance', 'EstimatedSalary']
to_typecast_object_to_num_cols_for_pipeline =  ['NumOfProducts', 'HasCrCard', 'IsActiveMember']

In [ ]:
def convert_object_to_int(x):
    return x.astype(int)


<h2 style="font-family: 'Brush Script MT', cursive; color: goldenrod;">Building the Preprocessor (Pipeline + ColumnTransformer)</h2>

In [ ]:
# Pipelines for each Type:

# Numeric Pipeline:

numeric_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler(with_mean=False))  # Change here to avoid centering sparse matrices
        ]
)

# Categorical Pipeline:
categorical_pipeline = Pipeline(
    steps=[
        ("ohe",OneHotEncoder(handle_unknown='ignore')),
        ("scaler",StandardScaler(with_mean=False))
    ]
)


object_to_num_pipeline = Pipeline(
    steps=[
        ("type_casting", FunctionTransformer(convert_object_to_int,validate=False)),
        ("scaler",StandardScaler(with_mean=False))
    ]
)


In [ ]:
# Combining all the Pipeline Components using the ColumnTransformer:

preprocessor = ColumnTransformer(transformers=[
    ('numeric_pipeline',numeric_pipeline,numerical_cols_for_pipeline),
    ('categorical_pipeline',categorical_pipeline,categorical_cols_for_pipeline),
    ('type_casting_pipeline',object_to_num_pipeline,to_typecast_object_to_num_cols_for_pipeline)
])

<h2 style="font-family: 'Brush Script MT', cursive; color: goldenrod;">Checking Class Imbalance in the Dataset</h2>

In [ ]:
# Checking for the Imbalance in the Target Feature:

def check_imbalance(y):
    print("Checking the class distribution:")
    print(y.value_counts(normalize=True))
    
    sns.countplot(x=y)
    plt.title("Class Distribution of the Target Column : Exited")
    plt.show()
    
    ratio = y.value_counts(normalize=True).values
    is_imbalanced = ratio[0] > 0.7 or ratio[1] > 0.7

    if is_imbalanced:
        print("Yes, the data is imbalanced.")
    else:
        print("No, the data is fairly balanced.")
    
    return is_imbalanced

In [ ]:
# Returns True if there is a significant class imbalance, where one class constitutes more than 70% of the data
# and the other class is less than 30%. In such cases, we apply SMOTE (Synthetic Minority Over-sampling Technique)
# to balance the classes. For more balanced distributions like 70:30, SMOTE might not be needed unless specifically
# chosen for model performance improvement.

check_imbalance(y)

<div style="font-family:Georgia; font-size:16px; border-left:5px solid purple; padding:10px; background-color:lightyellow; color:darkblue;">

<b>This means:</b><br><br>

<b>•</b> Class 0 accounts for <b>79.63%</b> of the training data  
<b>•</b> Class 1 accounts for <b>20.37%</b> of the training data  

<br>
<b>Conclusion:</b><br>
This dataset is <b>imbalanced</b> because one class (likely the negative class, 0) dominates the dataset.  
Since Class 1 (20.37%) is significantly underrepresented, the model might be biased toward predicting the majority class (Class 0) unless handled properly.

<br>

<b>Action:</b><br>
Apply <b>SMOTE</b> to oversample the minority class (1) and balance the data before training your classifiers.

</div>


<h2 style="font-family: 'Brush Script MT', cursive; color: goldenrod;">Splitting Data into Train, Validation, and Test Sets</h2>

In [ ]:
# First split into training+validation and test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Now split X_temp into train and validation
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp)

# Why 0.25? 
# Because 0.25 * 0.8 = 0.2
# So overall:
# 60% train, 20% val, 20% test

# Check shapes
print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")
print(f"X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")



<h2 style="font-family: 'Brush Script MT', cursive; color: goldenrod;">Saving the Raw Splits for Future Use</h2>

In [ ]:
independent_cols = list(X.columns)
dependent_cols = ['Exited']

overall_cols = independent_cols + dependent_cols

In [ ]:
# Combine features + target before saving
train_data = pd.DataFrame(np.c_[X_train, y_train], columns=overall_cols)
val_data = pd.DataFrame(np.c_[X_val, y_val], columns=overall_cols)
test_data = pd.DataFrame(np.c_[X_test, y_test], columns=overall_cols)


# Define the output directory
output_dir = os.path.join('..', 'Artifacts', 'Datasets')
os.makedirs(output_dir, exist_ok=True)  # create if not exists

# Define full paths for each file
train_path = os.path.join(output_dir, 'train.csv')
val_path = os.path.join(output_dir, 'validate.csv')
test_path = os.path.join(output_dir, 'test.csv')

# Save the splits
train_data.to_csv(train_path, index=False)
val_data.to_csv(val_path, index=False)
test_data.to_csv(test_path, index=False)

print(f"Files saved:\n- {train_path}\n- {val_path}\n- {test_path}")

In [ ]:
splits = {
    "X_train": X_train,
    "X_validation":X_val,
    "X_test": X_test,
    "y_train": y_train,
    "y_validation":y_val,
    "y_test": y_test
}

for name, data in splits.items():
    print(f"{name} → Shape: {data.shape}, Type: {type(data)}")


<h2 style="font-family: 'Brush Script MT', cursive; color: goldenrod;">Fitting and Transforming Training and Validation Data</h2>


In [ ]:
X_train = preprocessor.fit_transform(X_train)

In [ ]:
X_val = preprocessor.transform(X_val)


<h2 style="font-family: 'Brush Script MT', cursive; color: goldenrod;">Applying SMOTE to Balance the Training Data</h2>

In [ ]:
# Apply SMOTE (only on the training set): Apply SMOTE on X_train and y_train to balance the classes.

smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

In [ ]:
resampled = {
    "X_resampled": X_resampled,
    "y_resampled": y_resampled
}

for name, data in resampled.items():
    print(f"{name} → Shape: {data.shape}, Type: {type(data)}")


<div style="font-family:Georgia; font-size:16px; border-left:5px solid purple; padding:10px; background-color:lightyellow; color:darkblue;">
    <b>Preprocessing:</b><br>
    <br>
    Fit the preprocessor <b>only on the training data (X_train)</b>. This is because the preprocessor needs to learn from the training data (e.g., scaling, encoding).<br>
    <br>
    Transform <b>X_train</b> using <b>fit_transform()</b> to apply the transformations to the training set.<br>
    <br>
    Transform <b>X_val</b> and <b>X_test</b> using <b>transform()</b> since they should not influence the fitting process.
</div>


<h2 style="font-family: 'Brush Script MT', cursive; color: goldenrod;">Saving the Preprocessor</h2>


In [ ]:
# Setting the preprocessor save path
preprocessor_dir = os.path.join('..', 'Artifacts', 'Preprocessor')
preprocessor_path = os.path.join(preprocessor_dir, 'preprocessor_object.pkl')

# Create the directory if it doesn't exist
os.makedirs(preprocessor_dir, exist_ok=True)

# Save the preprocessor
with open(preprocessor_path, 'wb') as f:
    dill.dump(preprocessor, f)

print(f"\nPreprocessor saved successfully at: {preprocessor_path}")

<div style="font-family:Georgia; font-size:16px; border-left:5px solid purple; padding:10px; background-color:lightyellow; color:darkblue;">
    <b>Note on SMOTE and Data Splits:</b><br>
    <br>
    Apply <b>SMOTE only to X_train</b> (training data), not X_val or X_test.<br>
    <br>
    <b>Why?</b> SMOTE is used to balance the classes in the training data, helping the model learn better from imbalanced data.<br>
    <br>
    <b>Avoid data leakage:</b> Applying SMOTE to X_val or X_test would alter their real-world distribution and give overly optimistic results.
</div>


<h2 style="font-family: 'Brush Script MT', cursive; color: goldenrod;">Declaring Models and Their Hyperparameters</h2>

In [ ]:
models = {
    "Logistic_Regression": LogisticRegression(),
    "Support_Vector_Classifier": SVC(),
    "Decision_Tree_Classifier": DecisionTreeClassifier(),
    "Random_Forest_Classifier": RandomForestClassifier(),
    "XG_Boost_Classifier": XGBClassifier()
}


In [ ]:
params = {
    "Logistic_Regression": {
        "C": [0.1, 1, 10],
       "solver": ["liblinear", "lbfgs"],
        "max_iter": [100, 500, 1000]
    },
    
    "Support_Vector_Classifier": {
        "C": [0.1, 1, 10],
        "kernel": ["linear", "rbf", "poly"],
        "gamma": ["scale", "auto"]
    },
    
    "Decision_Tree_Classifier": {
        "criterion": ["gini", "entropy"],
        "max_depth": [None, 5, 10, 20],
        "min_samples_split": [2, 5, 10]
    },
    
    "Random_Forest_Classifier": {
        "n_estimators": [100, 200],
        "max_depth": [None, 5, 10],
        "min_samples_split": [2, 5],
        "bootstrap": [True, False]
    },
    
    "XG_Boost_Classifier": {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.01, 0.1, 0.2],
        "subsample": [0.8, 1.0]
    }
}


<div style="font-family:Georgia; font-size:16px; border-left:5px solid purple; padding:10px; background-color:lightyellow; color:darkblue;">

**Note:**  
  
 Accuracy alone isn't enough.  
 A model might have high accuracy but still perform poorly on minority classes.  
  
 So, let's define a custom scoring logic that considers:  
 
 - **Accuracy**
 - **F1-score**
 - **Precision**
 - **Recall**
 - **Confusion Matrix Quality** (where **low False Positives (FP)** and **low False Negatives (FN)** are seen as positive traits)

Focusing on these metrics gives a much fairer evaluation, especially in imbalanced datasets.
</div>



<h2 style="font-family: 'Brush Script MT', cursive; color: goldenrod;">Model Training and Evaluation</h2>

In [ ]:
results = []
best_model = None
best_score = -np.inf  # maximize custom score

for name, model in models.items():
    print(f"Training {name}...")

    # GridSearchCV on training data, evaluated using validation data
    clf = GridSearchCV(model, params[name], cv=5, scoring='accuracy', n_jobs=-1)
    clf.fit(X_train, y_train)  # <<< use training set

    y_val_pred = clf.predict(X_val)  # <<< predict on validation set

    # Metrics on validation set
    acc = accuracy_score(y_val, y_val_pred)
    precision = precision_score(y_val, y_val_pred, average='weighted')
    recall = recall_score(y_val, y_val_pred, average='weighted')
    f1 = f1_score(y_val, y_val_pred, average='weighted')

    try:
        if len(set(y_val)) > 2:
            roc = roc_auc_score(y_val, clf.predict_proba(X_val), multi_class='ovr')
        else:
            roc = roc_auc_score(y_val, clf.predict_proba(X_val)[:, 1])
    except:
        roc = None

    cm = confusion_matrix(y_val, y_val_pred)
    TN, FP, FN, TP = cm.ravel()

    # Custom scoring logic (based on validation)
    score = (
        acc * 0.3 +
        f1 * 0.3 +
        recall * 0.2 +
        precision * 0.1 +
        (TP / (TP + FN + 1e-6)) * 0.05 +  # Sensitivity
        (TN / (TN + FP + 1e-6)) * 0.05    # Specificity
    )

    results.append({
        'Model': name,
        'Best_Params': clf.best_params_,
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1_Score': f1,
        'ROC_AUC': roc,
        'Confusion_Matrix': cm,
        'Custom_Score': score
    })

    if score > best_score:
        best_score = score
        best_model = clf.best_estimator_


In [ ]:

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(by='Custom_Score', ascending=False , ignore_index=True)
results_df

In [ ]:
print("\nModel Ranking (Based on Custom Score):\n")
results_df[['Model', 'Accuracy', 'F1_Score', 'Precision', 'Recall', 'Custom_Score']]

In [ ]:
best_model

<h2 style="font-family: 'Brush Script MT', cursive; color: goldenrod;">Saving the Trained Model</h2>

In [ ]:
# Setting the model save path

model_dir = os.path.join('..','Artifacts','Model')
model_path = os.path.join(model_dir, 'Best_Model.pkl')

# Create the directory if it doesn't exist
os.makedirs(model_dir, exist_ok=True)

# Save the model
with open(model_path, 'wb') as f:
    dill.dump(best_model, f)

print(f"\nBest model saved successfully at: {model_path}")

---------------------